In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 1: IMPORTS AND CONFIGURATION
# ══════════════════════════════════════════════════════════════════════

import os
import re
import html
import json
import time
import random
import pickle
import asyncio
from pathlib import Path
from datetime import datetime, timedelta, timezone
from collections import Counter

import numpy as np
import pandas as pd

# ML / Data Science
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
import xgboost as xgb

# NLP
import nltk
from nltk.stem import WordNetLemmatizer
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import MaximalMarginalRelevance, KeyBERTInspired, PartOfSpeech
from umap import UMAP
from hdbscan import HDBSCAN

# Database
from sqlalchemy import create_engine, text, Column, Integer, String, Text, DateTime, Float, ForeignKey, UniqueConstraint,Boolean
from sqlalchemy.orm import declarative_base, relationship
from sqlalchemy.dialects.postgresql import insert as pg_insert

# Language detection
from lingua import Language, LanguageDetectorBuilder

# LLM
import litellm

# Data collection
from dotenv import load_dotenv
from pytrends_modern.request import TrendReq
from googleapiclient.discovery import build
from apify_client import ApifyClientAsync
from apify_client.errors import ApifyApiError
import numpy as np

# nltk.download("wordnet", quiet=True)
# nltk.download("omw-1.4", quiet=True)
# nltk.download("punkt", quiet=True)
nltk.download('stopwords', quiet=True)
print(" Libraries loaded")

In [ ]:
# ── Load configuration ──────────────────────────────────────────────
load_dotenv()

USE_EXISTING_DATA_from_DB = False
with open("keywords.json", "r") as f:
    KEYWORDS = json.load(f)

# ── Geography ───────────────────────────────────────────────────────
GEO = "IE"
YOUTUBE_REGION = "IE"
REDDIT_REGIONS = {"ireland", "irishfitfam", "irishfood"}
DAYS = 180
MONTHS = 6

# ── APIs ────────────────────────────────────────────────────────────
APIFY_TOKEN = os.getenv("APIFY_TOKEN", "")
APIFY_REDDIT_ACTOR = os.getenv("APIFY_REDDIT_ACTOR_ID", "")
APIFY_TIKTOK_ACTOR = os.getenv("APIFY_TIKTOK_ACTOR_ID", "")
YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "")
LLM_API_KEY = os.getenv("LLM_API_KEY", "")
LLM_MODEL = os.getenv("LLM_MODEL", "")
LLM_MODEL_FALLBACKS = os.getenv("LLM_MODEL_FALLBACKS", "gpt-4,gpt-3.5-turbo").split(",")

# ── Data collection params ──────────────────────────────────────────
GOOGLE_SEEDS = KEYWORDS["gtrends"]
GOOGLE_SLEEP = 3
GOOGLE_RETRIES = 3
FOOD_SUBREDDITS = KEYWORDS["reddit"]
REDDIT_LIMIT = 150
YOUTUBE_SEARCH_TERMS = KEYWORDS["youtube"]
YOUTUBE_COMMENT_MAX = 100
YOUTUBE_MAX = 50
TIKTOK_SEARCH_TERMS = KEYWORDS["tiktok"]
REDDIT_SEARCH_TERMS = KEYWORDS["reddit"]
HASHTAG_SEARCH_TERMS = KEYWORDS["hashtags"]
APIFY_LIMIT = 1000
MAX_PER_REDDIT_KEYWORD = 30
MAX_PER_TIKTOK_KEYWORD = 30

# ── Model parameters ────────────────────────────────────────────────
GROWTH_THRESHOLD = 0.20
SUSTAIN_WEEKS = 2
SUSTAIN_MULTIPLIER = 1.2
HOLD_OUT_WEEKS = 2


# ── Other ───────────────────────────────────────────────────────────
NOISE_TOKENS = set(KEYWORDS.get("noise_tokens", []))

# ── Database URL ────────────────────────────────────────────────────
DB_URL = f"postgresql+psycopg2://{os.getenv('DB_USER', 'postgres')}:{os.getenv('DB_PASSWORD', '')}@{os.getenv('DB_HOST', 'localhost')}:{os.getenv('DB_PORT', 5432)}/{os.getenv('DB_NAME', 'postgres')}"

print(" Configuration loaded")


In [ ]:
print(DB_URL)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 2: DATABASE SCHEMA WITH ALL FEATURES
# ══════════════════════════════════════════════════════════════════════

Base = declarative_base()

class RawPost(Base):
    __tablename__ = "raw_posts"
    id = Column(Integer, autoincrement=True, primary_key=True)
    platform = Column(String)
    source = Column(String)
    title = Column(String)
    full_text = Column(Text)
    language = Column(String)
    created_date = Column(DateTime(timezone=True))
    week_start_date = Column(DateTime(timezone=True))
    source_url = Column(String, nullable=True)
    post_topic = relationship("PostTopic", back_populates="raw_post", uselist=False)
    __table_args__ = (UniqueConstraint("platform", "source", "title", "created_date"),)


class PostTopic(Base):
    __tablename__ = "posts_with_topics"
    id = Column(Integer, autoincrement=True, primary_key=True)
    raw_post_id = Column(Integer, ForeignKey("raw_posts.id"))
    platform = Column(String)
    source = Column(String)
    title = Column(String)
    full_text = Column(Text)
    week_start_date = Column(DateTime(timezone=True))
    topic_id = Column(Integer)
    food_category = Column(String)
    raw_post = relationship("RawPost", back_populates="post_topic")


class WeeklySnapshot(Base):
    __tablename__ = "weekly_snapshots"
    id = Column(Integer, autoincrement=True, primary_key=True)
    food_category = Column(String, nullable=False)
    week_start_date = Column(DateTime(timezone=True), nullable=False)


    total_posts = Column(Integer,nullable=True)
    growth_rate = Column(Float,nullable=False)
    platform_count = Column(Integer, nullable=False, default=0)
    sustained_growth = Column(Integer,nullable=False,default=0)
    ratio_to_peak = Column(Float,nullable=False)
    rolling_avg = Column(Float,nullable=False)
    ra4 = Column(Float,nullable=False)
    rank_wk = Column(Float,nullable=False)

    #non feature columns for future predictions
    future_avg  = Column(Float,nullable=True)
    # Target
    will_trend = Column(Boolean, nullable=True)
    __table_args__ = (UniqueConstraint("food_category", "week_start_date"),)


class TrendPrediction(Base):
    __tablename__ = "trend_predictions"
    id = Column(Integer, autoincrement=True, primary_key=True)
    food_category = Column(String)

    #features
    total_posts = Column(Integer,nullable=False)
    growth_rate = Column(Float,nullable=False)
    platform_count = Column(Integer, nullable=False, default=0)
    sustained_growth = Column(Integer,nullable=False,default=0)
    ratio_to_peak = Column(Float,nullable=False)
    rolling_avg = Column(Float,nullable=False)
    ra4 = Column(Float,nullable=False)
    rank_wk = Column(Float,nullable=False)
    
    #features
    trend_probability = Column(Float)
    will_trend = Column(Boolean)
    prediction_week = Column(DateTime(timezone=True))


# ── Connect to DB ───────────────────────────────────────────────────
try:
    engine = create_engine(DB_URL, connect_args={"connect_timeout": 10})
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version()")).fetchone()
        print(f" Connected: {result[0][:55]}...")
    DB_AVAILABLE = True
except Exception as e:
    print(f" Database unavailable: {e}")
    DB_AVAILABLE = False
    engine = None

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 2B: CREATE TABLES
# ══════════════════════════════════════════════════════════════════════

if DB_AVAILABLE and engine is not None:
    Base.metadata.create_all(bind=engine)
    print(" Tables ensured: raw_posts, posts_with_topics, weekly_snapshots, trend_predictions")
else:
    print(" Skipped table creation because database connection is unavailable.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 3: DATABASE HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════════

# All snapshot columns
SNAPSHOT_COLS = ["food_category","week_start_date",
                 "total_posts", "platform_count", "growth_rate", 
                 "sustained_growth", "ratio_to_peak", "rolling_avg", 
                 'future_avg',"will_trend"]

# All prediction columns
PREDICTION_COLS = [
    "food_category","growth_rate", 
    "platform_count", "sustained_growth", "ratio_to_peak",
    "rolling_avg","trend_probability", "will_trend", "prediction_week"
]


def clear_table(name):
    if not DB_AVAILABLE:
        return
    with engine.connect() as conn:
        conn.execute(text(f'TRUNCATE TABLE "{name}" RESTART IDENTITY CASCADE'))
        conn.commit()


def write_to_db(df, table, if_exists="append"):
    if not DB_AVAILABLE or df.empty:
        return
    try:
        if if_exists == "replace":
            clear_table(table)
            if_exists = "append"
        df.to_sql(table, engine, if_exists=if_exists, index=False)
        print(f"   {len(df)} rows → '{table}'")
    except Exception as e:
        print(f"   Write failed '{table}': {e}")


def read_from_db(query):
    if not DB_AVAILABLE:
        return pd.DataFrame()
    try:
        return pd.read_sql(query, engine)
    except Exception as e:
        print(f"  Query failed: {e}")
        return pd.DataFrame()


def upsert_raw_posts(df):
    if not DB_AVAILABLE or df.empty:
        return
    cols = ["platform", "source", "title", "full_text",
            "created_date", "week_start_date", "language", "source_url"]
    records = df[cols].to_dict(orient="records")
    stmt = pg_insert(RawPost).values(records)
    stmt = stmt.on_conflict_do_update(
        constraint="raw_posts_platform_source_title_created_date_key",
        set_={"language": stmt.excluded.language, "source_url": stmt.excluded.source_url}
    )
    try:
        with engine.begin() as conn:
            conn.execute(stmt)
        print(f"   {len(records)} rows upserted → raw_posts")
    except Exception as e:
        print(f"   Upsert failed: {e}")


def upsert_weekly_snapshots(df):
    if not DB_AVAILABLE or df.empty:
        return
    
    # Ensure all columns exist
    for col in SNAPSHOT_COLS:
        if col not in df.columns:
            df[col] = None
    
    records = df[SNAPSHOT_COLS].to_dict(orient="records")
    stmt = pg_insert(WeeklySnapshot).values(records)
    stmt = stmt.on_conflict_do_update(
        constraint="weekly_snapshots_food_category_week_start_date_key",
        set_={c: getattr(stmt.excluded, c) for c in SNAPSHOT_COLS
              if c not in ("food_category", "week_start_date")}
    )
    try:
        with engine.begin() as conn:
            conn.execute(stmt)
        print(f"   {len(records)} rows upserted → weekly_snapshots")
    except Exception as e:
        print(f"   Upsert failed: {e}")


print(" DB helpers ready")

In [ ]:
def fetch_google_trends_rising(seed_topics, geo=GEO):
    """Fetch rising + top related queries for each seed. Retries on 429."""
    pytrends = TrendReq(hl="en-IE", tz=0, retries=2, backoff_factor=0.5)
    end_date = datetime.today()
    start_date = end_date - timedelta(days=180)  # ~6 months
    # Format as 'YYYY-MM-DD YYYY-MM-DD'
    timeframe = f"{start_date.strftime('%Y-%m-%d')} {end_date.strftime('%Y-%m-%d')}"
    records = []
    for topic in seed_topics:
        for attempt in range(GOOGLE_RETRIES):
            try:
                pytrends.build_payload([topic], timeframe = timeframe, geo=geo)
                related = pytrends.related_queries()
                for qtype in ["top", "rising"]:
                    df = related.get(topic, {}).get(qtype)
                    if df is not None and not df.empty:
                        df = df.copy()
                        df["seed_topic"]     = topic
                        df["query_type"]     = qtype
                        df["collected_date"] = pd.Timestamp.today().normalize()
                        records.append(df)
                time.sleep(random.uniform(GOOGLE_SLEEP, 10))
                break   # success — move to next topic
            except Exception as e:
                if "429" in str(e) and attempt < GOOGLE_RETRIES - 1:
                    wait = GOOGLE_SLEEP * (attempt + 2)
                    print(f"  429 on '{topic}' — waiting {wait}s then retrying...")
                    time.sleep(wait)
                else:
                    print(f"  Warning: '{topic}': {e}")
                    break
    return pd.concat(records, ignore_index=True) if records else pd.DataFrame()

In [ ]:
def fetch_youtube_videos(
    search_terms,
    api_key,
    max_results=YOUTUBE_MAX,
    max_comments=YOUTUBE_COMMENT_MAX
):

    youtube = build("youtube", "v3", developerKey=api_key)

    published_after = (
        datetime.now(timezone.utc) - timedelta(days=DAYS)
    ).strftime("%Y-%m-%dT%H:%M:%SZ")

    video_records = []
    comment_records = []

    for term in search_terms:
        try:
            response = youtube.search().list(
                q=term,
                part="snippet",
                type="video",
                maxResults=max_results,
                publishedAfter=published_after,
                relevanceLanguage="en",
                regionCode=YOUTUBE_REGION,
                order="viewCount"
            ).execute()

            for item in response.get("items", []):

                video_id = item["id"]["videoId"]
                sn = item["snippet"]


                video_records.append({
                    "platform": "YouTube",
                    "source": f"YouTube:{term}",
                    "search_term": term,
                    "video_id": video_id,
                    "title": sn.get("title", ""),
                    "description": sn.get("description", ""),
                    "full_text": f"{sn.get('title','')} {sn.get('description','')}",
                    "channel": sn.get("channelTitle", ""),
                    "created_date": pd.Timestamp(sn.get("publishedAt")).normalize(),
                    "source_url": f"https://www.youtube.com/watch?v={video_id}"
                })

                # Fetch comments

                try:
                    comments = youtube.commentThreads().list(
                        part="snippet",
                        videoId=video_id,
                        maxResults=max_comments,
                        order="relevance",
                        textFormat="plainText"
                    ).execute()

                    for c in comments.get("items", []):
                        csn = c["snippet"]["topLevelComment"]["snippet"]
                        comment_text = csn.get("textDisplay", "")

                        comment_records.append({
                            "video_id": video_id,
                            "search_term": term,
                            "video_title": sn.get("title", ""),
                            "comment": comment_text,
                            "author": csn.get("authorDisplayName", ""),
                            "likes": csn.get("likeCount", 0),
                            "source_url": f"https://www.youtube.com/watch?v={video_id}&lc={c['id']}",
                            "comment_date": pd.Timestamp(
                                csn.get("publishedAt")
                            ).normalize()
                        })

                except Exception:
                    pass

                time.sleep(0.1)

        except Exception as e:
            print(f"Warning: YouTube '{term}': {e}")

    videos_df = pd.DataFrame(video_records)
    comments_df = pd.DataFrame(comment_records)

    return videos_df, comments_df

In [ ]:
if not APIFY_TOKEN or not APIFY_REDDIT_ACTOR or not APIFY_TIKTOK_ACTOR:
    print("Apify credentials missing --- skipping Apify data collection.")
    print("Set APIFY_TOKEN, APIFY_REDDIT_ACTOR_ID and APIFY_TIKTOK_ACTOR_ID in .env")
    client = None
else:
    client = ApifyClientAsync(APIFY_TOKEN)
    print("Apify client initialised.")
tiktok_raw = pd.DataFrame()

posted_after = (datetime.now(timezone.utc) - timedelta(days=DAYS)).strftime("%Y-%m-%dT%H:%M:%SZ")

In [ ]:
async def scrape_reddit(keyword, max_posts=MAX_PER_REDDIT_KEYWORD,posted_after=posted_after):
    if client is None:
        return []
    
    run_input = {
        "searches": [keyword],
        "searchPosts": True,
        "searchComments": True,
        "searchCommunities": False,
        "withinCommunity": "",
        "searchSort": "new",
        "searchTime": "year", 
        "startUrls": [{"url": f"https://www.reddit.com/r/{keyword}/"}],
        "fastMode": True,
        "subredditUrls": [],
        "postedAfter": posted_after,
        "postedBefore": "", 
        "commentedAfter": "",
        "commentedBefore": "", 
        "onlyWithFlair": False,
        "crawlCommentsPerPost": True,
        "includeNSFW": False,
        "maxPostsCount": max_posts,
        "maxCommentsCount": 10,
        "maxCommentsPerPost": 10,
        "maxCommunitiesCount": 2,
        "mcpConnector": None,
        "mcpMode": "perPost",
        "mcpTarget": None,
        "mcpComments": "ignore",
        "mcpCommentsPerPost": 5,
        "mcpMessage": """*{{title}}*{{postUrl}}""",
        "mcpTool": None,
        "mcpArguments": None,
        "mcpMaxItems": 50,
        "mcpServerUrl": None,
        "mcpServerToken": None,
        "proxy": {
            "useApifyProxy": True,
            "apifyProxyGroups": ["RESIDENTIAL"],
        },
    }
    
    run_input = {k: v for k, v in run_input.items() if v is not None}

    run = await client.actor(APIFY_REDDIT_ACTOR).call(run_input=run_input)
    if run is None:
        print(f"Warning: Reddit actor returned None for keyword '{keyword}'")
        return []
    dataset_id = run["defaultDatasetId"] if isinstance(run, dict) else run.default_dataset_id
    dataset = client.dataset(dataset_id)
    rows = []
    async for item in dataset.iterate_items():
        body_text = item.get("text") or item.get("body") or item.get("selftext") or ""
        comments_value = item.get("comments") or item.get("commentData") or item.get("topComments") or item.get("latestComments") or []
        if isinstance(comments_value, list):
            comments_text = "\n".join(
                str(comment.get("text") or comment.get("body") or comment.get("comment") or comment.get("content") or "")
                for comment in comments_value
                if isinstance(comment, dict)
            )
        else:
            comments_text = str(comments_value) if comments_value else ""
        rows.append({
            "platform": "Reddit",
            "source": "Reddit:" + str(keyword),
            "title": item.get("title", ""),
            "description": item.get("text") or item.get("body") or item.get("selftext") or "",
            "full_text": f"{item.get('title', '')} {body_text} {comments_text}".strip(),
            "comments": comments_text,
            "subreddit": item.get("subredditName"),
            "score": item.get("score"),
            "url": item.get("url"),
            "created": item.get("createdAt")
        })
    return rows

In [ ]:

async def scrape_tiktok(keyword, max_videos=MAX_PER_TIKTOK_KEYWORD, posted_after=posted_after):
    if client is None:
        return []
    
    run_input = {
        "hashtags": [keyword],
        "resultsPerPage": max_videos,
        "profiles": None,
        "profileScrapeSections": ["videos"],
        "profileSorting": "latest",
        "excludePinnedPosts": False,
        "oldestPostDateUnified": posted_after,
        "newestPostDate": "", 
        "mostDiggs": None,
        "leastDiggs": None,
        "maxFollowersPerProfile": 0,
        "maxFollowingPerProfile": 0,
        "searchQueries": None,
        "searchSection": "",
        "maxProfilesPerQuery": 10,
        "videoSearchSorting": "MOST_RELEVANT",
        "videoSearchDateFilter": "LAST_3_MONTHS",
        "scrapeRelatedSearchWords": False,
        "postURLs": None,
        "scrapeRelatedVideos": False,
        "scrapeAdditionalAuthorMeta": False,
        "shouldDownloadVideos": False,
        "shouldDownloadCovers": False,
        "shouldDownloadSlideshowImages": False,
        "shouldDownloadAvatars": False,
        "shouldDownloadMusicCovers": False,
        "videoKvStoreIdOrName": None,
        "downloadSubtitlesOptions": "NEVER_DOWNLOAD_SUBTITLES",
        "commentsPerPost": 10,
        "topLevelCommentsPerPost": 10,
        "maxRepliesPerComment": 5,
        "proxyCountryCode": "IE",
    }
    run_input = {k: v for k, v in run_input.items() if v is not None}
    run = await client.actor(APIFY_TIKTOK_ACTOR).call(run_input=run_input)
    if run is None:
        print(f"Warning: TikTok actor returned None for keyword '{keyword}'")
        return []
    dataset_id = run["defaultDatasetId"] if isinstance(run, dict) else run.default_dataset_id
    dataset = client.dataset(dataset_id)
    rows = []
    async for item in dataset.iterate_items():
        body_text = item.get("text") or item.get("desc") or item.get("description") or ""
        comments_value = item.get("comments") or item.get("commentList") or item.get("commentData") or item.get("commentItems") or []
        if isinstance(comments_value, list):
            comments_text = "\n".join(
                str(comment.get("text") or comment.get("commentText") or comment.get("body") or comment.get("content") or "")
                for comment in comments_value
                if isinstance(comment, dict)
            )
        else:
            comments_text = str(comments_value) if comments_value else ""
        rows.append({
            "platform": "TikTok",
            "source": "TikTok:" + str(keyword),
            "title": item.get("title") or body_text[:120],
            "description": body_text,
            "full_text": f"{item.get('title', '')} {body_text} {comments_text}".strip(),
            "comments": comments_text,
            "author": item.get("authorMeta", {}).get("name"),
            "likes": item.get("diggCount"),
            "shares": item.get("shareCount"),
            "views": item.get("playCount"),
            "hashtags": ",".join(
                h.get("name", "")
                for h in item.get("hashtags", [])
            ),
            "created": item.get("createTime")
        })
    return rows

In [ ]:
import pickle
from pathlib import Path
from apify_client.errors import ApifyApiError
import asyncio

reddit_raw = pd.DataFrame() 
tiktok_raw = pd.DataFrame()

# ── Checkpointing setup ──────────────────────────────────────────────
CHECKPOINT_DIR = Path("apify_checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
REDDIT_CKPT = CHECKPOINT_DIR / "reddit_raw.pkl"
TIKTOK_CKPT = CHECKPOINT_DIR / "tiktok_raw.pkl"

def load_checkpoint(path, fallback_df):
    if path.exists():
        try:
            df = pickle.load(open(path, "rb"))
            print(f"  Resumed {len(df)} rows from {path.name}")
            return df
        except Exception as e:
            print(f"  Could not load checkpoint {path.name}: {e}")
    return fallback_df

def save_checkpoint(df, path):
    try:
        pickle.dump(df, open(path, "wb"))
    except Exception as e:
        print(f"  Warning: failed to save checkpoint {path.name}: {e}")

# Resume from any previous partial run before doing anything else
reddit_raw = load_checkpoint(REDDIT_CKPT, reddit_raw)
tiktok_raw = load_checkpoint(TIKTOK_CKPT, tiktok_raw)

# ── Rate limit detection ─────────────────────────────────────────────
def is_rate_limit_error(e):
    if isinstance(e, ApifyApiError):
        return getattr(e, "status_code", None) == 429
    msg = str(e).lower()
    return any(s in msg for s in [
        "429", "rate limit", "too many requests",
        "usage hard limit", "monthly usage", "quota exceeded"
    ])

# ── Retry with exponential backoff ───────────────────────────────────
async def call_with_backoff(fn, *args, max_retries=3, base_wait=10, **kwargs):
    for attempt in range(max_retries):
        try:
            return await fn(*args, **kwargs), False
        except Exception as e:
            if "usage hard limit" in str(e).lower() or "monthly usage" in str(e).lower():
                print(f"  Monthly Apify usage limit hit — no point retrying until next cycle.")
                return None, True   # immediately signal rate-limited, skip retries
            if is_rate_limit_error(e):
                wait = base_wait * (2 ** attempt)
                print(f"  Rate limited — waiting {wait}s (attempt {attempt+1}/{max_retries})")
                await asyncio.sleep(wait)
            else:
                raise
    return None, True

# ── Per-keyword scrape wrappers (save checkpoint after each success) ──
async def try_scrape_reddit(keyword):
    global reddit_raw
    if client is None:
        return False
    reddit_needed = APIFY_LIMIT - len(reddit_raw)
    posts_to_fetch = min(reddit_needed, MAX_PER_REDDIT_KEYWORD)
    print(f"Searching Reddit for '{keyword}' (Fetching {posts_to_fetch} posts)")
    try:
        data, limited = await call_with_backoff(scrape_reddit, keyword, max_posts=posts_to_fetch)
        if limited:
            print(f"  Reddit still rate-limited after retries — keeping {len(reddit_raw)} rows collected so far.")
            return True
        if data:
            data = data[:reddit_needed]
            new_df = pd.DataFrame(data)
            reddit_raw = new_df if reddit_raw.empty else pd.concat([reddit_raw, new_df], ignore_index=True)
            save_checkpoint(reddit_raw, REDDIT_CKPT)
        print(f"  Retrieved {len(data)} posts. Reddit total: {len(reddit_raw)}/{APIFY_LIMIT}")
        return False
    except Exception as e:
        print("  Reddit Error:", e)
        return False

async def try_scrape_tiktok(keyword):
    global tiktok_raw
    if client is None:
        return False
    tiktok_needed = APIFY_LIMIT - len(tiktok_raw)
    videos_to_fetch = min(tiktok_needed, MAX_PER_TIKTOK_KEYWORD)
    print(f"Searching TikTok for '{keyword}' (Fetching {videos_to_fetch} videos)")
    try:
        data, limited = await call_with_backoff(scrape_tiktok, keyword, max_videos=videos_to_fetch)
        if limited:
            print(f"  TikTok still rate-limited after retries — keeping {len(tiktok_raw)} rows collected so far.")
            return True
        if data:
            data = data[:tiktok_needed]
            new_df = pd.DataFrame(data)
            tiktok_raw = new_df if tiktok_raw.empty else pd.concat([tiktok_raw, new_df], ignore_index=True)
            save_checkpoint(tiktok_raw, TIKTOK_CKPT)
        print(f"  Retrieved {len(data)} videos. TikTok total: {len(tiktok_raw)}/{APIFY_LIMIT}")
        return False
    except Exception as e:
        print("  TikTok Error:", e)
        return False

# ── Interleaved collection so neither platform gets starved ──────────
async def run_apify_collection(reddit_terms, tiktok_terms):
    reddit_terms, tiktok_terms = list(reddit_terms), list(tiktok_terms)

    async def reddit_worker():
        for kw in reddit_terms:
            if len(reddit_raw) >= APIFY_LIMIT:
                break
            limited = await try_scrape_reddit(kw)
            if limited:
                print("Reddit rate-limited — stopping Reddit worker.")
                break
            await asyncio.sleep(2)

    async def tiktok_worker():
        for kw in tiktok_terms:
            if len(tiktok_raw) >= APIFY_LIMIT:
                break
            limited = await try_scrape_tiktok(kw)
            if limited:
                print("TikTok rate-limited — stopping TikTok worker.")
                break
            await asyncio.sleep(2)

    await asyncio.gather(reddit_worker(), tiktok_worker())

In [ ]:
from lingua import Language, LanguageDetectorBuilder

_detector = LanguageDetectorBuilder.from_all_languages() \
    .with_preloaded_language_models() \
    .build()

def detect_language(text):
    """Returns ISO 639-1 language code, or 'unknown' on failure."""
    if not text or not text.strip():
        return "unknown"
    
    try:
        language = _detector.detect_language_of(text.strip())
        if language is None:
            return "unknown"
        
        # Convert to string, split by '.', and take the last piece (e.g., 'EN'), then lowercase it
        return str(language.iso_code_639_1).split('.')[-1].lower()
        
    except Exception as e:
        print(e)
        return "unknown"

SCHEMA_COLS = ["platform", "source", "title", "description", "comments", "full_text", "created_date"]
sources = []

reddit_raw = pd.DataFrame()
gtrends_raw = pd.DataFrame()
youtube_videos_df = pd.DataFrame()
youtube_comments_df = pd.DataFrame()
tiktok_raw = pd.DataFrame()

if not USE_EXISTING_DATA_from_DB:
    # ── Collect data from all sources ─────────────────────────────────────
    print("Collecting data from all sources...")
    gtrends_raw = fetch_google_trends_rising(GOOGLE_SEEDS, geo=GEO)
    print(f"  Google Trends: {len(gtrends_raw)} rows")

    youtube_videos_df, youtube_comments_df = fetch_youtube_videos(
        YOUTUBE_SEARCH_TERMS, api_key=YOUTUBE_API_KEY,
        max_results=YOUTUBE_MAX, max_comments=YOUTUBE_COMMENT_MAX
    )
    print(f"  YouTube videos: {len(youtube_videos_df)} rows")
    print(f"  YouTube comments: {len(youtube_comments_df)} rows")

    #asyncio.run(run_apify_collection(REDDIT_SEARCH_TERMS, TIKTOK_SEARCH_TERMS))
    #print(f"  Reddit: {len(reddit_raw)} rows")
    #print(f"  TikTok: {len(tiktok_raw)} rows")
# Dedupe YouTube videos by video_id — same video can surface under
    if not youtube_videos_df.empty and "video_id" in youtube_videos_df.columns:
            before = len(youtube_videos_df)
            youtube_videos_df = youtube_videos_df.drop_duplicates(subset=["video_id"], keep="first")
            print(f"  YouTube video dedup: {before} -> {len(youtube_videos_df)}")

    if not youtube_comments_df.empty and "video_id" in youtube_comments_df.columns:
            before = len(youtube_comments_df)
            youtube_comments_df = youtube_comments_df.drop_duplicates(subset=["video_id", "comment"], keep="first")
            print(f"  YouTube comment dedup: {before} -> {len(youtube_comments_df)}")

    for raw_df, name in [
        (reddit_raw,  "Reddit"),
        (youtube_videos_df, "YouTube"),
        (youtube_comments_df, "YouTube Comments"),
        (tiktok_raw, "TikTok"),
    ]:
        if not raw_df.empty:
            df = raw_df.copy()
            if name == "YouTube Comments":
                df = df.rename(columns={"comment": "full_text", "video_title": "title", "comment_date": "created_date"})
                df["platform"] = f"YouTube - Comments"
                df["source"] = df["search_term"].apply(lambda t: f"YouTube:{t}:comment")
                df["description"] = df["full_text"]
                df["comments"] = df["full_text"]
            elif name == "Reddit":
                df["description"] = df.get("description", df.get("full_text", ""))
                df["comments"] = df.get("comments", "")
            elif name == "TikTok":
                df["description"] = df.get("description", df.get("full_text", ""))
                df["comments"] = df.get("comments", "")
            df["created_date"] = pd.to_datetime(df.get("created_date", df.get("created", pd.Timestamp.today())))
            df["source_url"] = df.get("source_url", "")
            if isinstance(df["created_date"], pd.Series):
                df["created_date"] = df["created_date"].dt.tz_localize(None)
            else:
                df["created_date"] = pd.Timestamp(df["created_date"]).tz_localize(None)
            if "description" not in df.columns:
                df["description"] = df.get("full_text", "")
            if "comments" not in df.columns:
                df["comments"] = ""
            if "full_text" not in df.columns:
                if name == "YouTube Comments":
                    df["full_text"] = df["comment"].fillna("").astype(str)
                else:
                    df["full_text"] = df["title"].fillna("").astype(str) + " " + df["description"].fillna("").astype(str) + " " + df["comments"].fillna("").astype(str)
            
            sources.append(df.reindex(columns=SCHEMA_COLS + [c for c in df.columns if c not in SCHEMA_COLS]))
            print(f"  Added {name}: {len(df)} rows")

    if not gtrends_raw.empty:
        rising = gtrends_raw[gtrends_raw["query_type"] == "rising"].copy()
        rising["platform"]       = "Google Trends"
        rising["source"]         = "Google Trends Rising"
        rising["title"]          = rising["query"]
        rising["description"]    = rising["query"]
        rising["comments"]       = ""
        rising["full_text"]      = rising["query"]
        rising["created_date"]   = pd.to_datetime(rising.get("collected_date", pd.Timestamp.today())).dt.tz_localize(None)
        sources.append(rising.reindex(columns=SCHEMA_COLS))
        print(f"  Added Google Trends rising: {len(rising)} rows")

    all_texts_df = pd.concat(sources, ignore_index=True)
    all_texts_df["created_date"]    = pd.to_datetime(all_texts_df["created_date"]).dt.tz_localize(None)
    all_texts_df["week_start_date"] = all_texts_df["created_date"].dt.to_period("W").dt.start_time
    all_texts_df = all_texts_df.dropna(subset=["full_text"])
    all_texts_df = all_texts_df[all_texts_df["full_text"].str.strip() != ""].reset_index(drop=True)
    all_texts_df = all_texts_df.drop_duplicates(
        subset=["platform", "source", "title", "created_date"]
    ).reset_index(drop=True)

    # -- Language detection on ALL rows (so raw_posts gets real values) --------
    print("  Detecting languages...")
    all_texts_df["language"] = all_texts_df["full_text"].apply(detect_language)

    # -- Upsert to DB (safe to re-run on same week -- no duplicate key crash) --
    if not all_texts_df.empty:
        upsert_raw_posts(
            all_texts_df.dropna(subset=["created_date", "week_start_date"])[
                ["platform", "source", "title", "full_text",
                "created_date", "week_start_date", "language","source_url"]
            ]
        )


In [ ]:
display(all_texts_df)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 4: LOAD EXISTING RAW POSTS FROM DATABASE
# ══════════════════════════════════════════════════════════════════════

# Load all English posts from DB
print("Loading existing English posts from database...")
raw_query = """
    SELECT id, platform, source, title, full_text, language,
        created_date, week_start_date, source_url
    FROM raw_posts 
    WHERE language = 'en'
    ORDER BY week_start_date DESC
"""
all_texts_df = read_from_db(raw_query)  

# all_texts_df = pd.read_excel("final_raw_posts.xlsx")
# all_texts_df = all_texts_df[all_texts_df["language"] == "en"]

if all_texts_df.empty:
    print(" No data in raw_posts table. Run data collection cells if needed.")
else:
    all_texts_df["created_date"] = pd.to_datetime(all_texts_df["created_date"], utc=True)
    all_texts_df["week_start_date"] = pd.to_datetime(all_texts_df["week_start_date"], utc=True)
    
    print(f" Loaded {len(all_texts_df)} English posts from database")
    print(f"   Platforms: {all_texts_df['platform'].nunique()}")
    print(f"   Weeks: {all_texts_df['week_start_date'].nunique()}")
    print(f"   Range: {all_texts_df['week_start_date'].min().date()} → {all_texts_df['week_start_date'].max().date()}")
    display(all_texts_df[["platform", "source", "title", "created_date", "week_start_date"]].head(5))

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 5: TOPIC DISCOVERY WITH BERTOPIC
# ══════════════════════════════════════════════════════════════════════
from nltk.corpus import stopwords
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
# ── Configuration ───────────────────────────────────────────────────
SEED_TOPICS = KEYWORDS.get("seed_topics", [])

CUSTOM_STOP_WORDS = list(CountVectorizer(stop_words="english").get_stop_words())
SEED_WORDS = sorted({w.strip().lower() for group in SEED_TOPICS for w in group})

FOOD_COMPOUNDS = KEYWORDS.get("food_compounds", [])

lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()


# ── Preprocess texts ────────────────────────────────────────────────
all_texts_df['clean_title'] = all_texts_df['title'].apply(clean_text)
all_texts_df["clean_full_text"] = all_texts_df['full_text'].apply(clean_text)

is_similar_title_and_text = all_texts_df["clean_title"].str.lower() == all_texts_df["clean_full_text"].str.lower()

all_texts_df["clean_combined_text"] = np.where(
    is_similar_title_and_text,
    all_texts_df["clean_full_text"].fillna(""),
    all_texts_df["title"].fillna("").astype(str) + " " + all_texts_df["full_text"].fillna("").astype(str)
)

all_texts_df = all_texts_df[all_texts_df['clean_combined_text'].str.split().str.len() >= 5]  

# Food filter - broader for smaller dataset
food_keywords = [
    'food','eat','restaurant','recipe','cook','dish','meal','dinner','lunch',
    'breakfast','brunch','cafe','coffee','tea','vegan','vegetarian','organic',
    'gluten','dairy','sourdough','ferment','kombucha','matcha','avocado','foodie',
    'delicious','tasty','yummy','cuisine','ingredient','bake','grill','fry','roast',
    'menu','takeaway','delivery','deliveroo','just eat','protein','smoothie','bowl',
    'salad','burger','pizza','sushi','ramen','thai','indian','chinese','italian',
    'mexican','stew','coddle','boxty','colcannon','chowder','guinness','whiskey',
    'craft beer','gin','cocktail','pub grub','chipper','spice bag','fillet roll',
    'deli','superfood','keto','paleo','plant-based','plant based','oat milk',
    'almond milk','snack','dessert','pastry','chocolate','ice cream','gelato',
    'farmers market','sustainable','locally sourced','tapas','wine','bar','bistro',
    'baker','butcher','cheese','bread','pasta','noodle','curry','kebab','wrap',
    'seafood','fish','chicken','beef','lamb','pork','tofu'
] + SEED_WORDS + FOOD_COMPOUNDS

pattern = r'\b(' + '|'.join(food_keywords) + r')\b'

def get_food_entities(text):
    # if len(text.split()) < 4:
    #     return []
    try:
        ents = ner_pipe(text[:512]) # Truncate long posts for speed
        # 0.7 threshold is due to high noise characters in social media above 80% will be aggressive  for foods
        return [e['word'].strip().lower() for e in ents if e['score'] > 0.70]
    except Exception:
        return []

df_food = all_texts_df[
    all_texts_df['clean_combined_text'].str.contains(pattern, case=False, na=False, regex=True)
].copy()
df_food = df_food.reset_index(drop=True)
# ner_tokenizer = AutoTokenizer.from_pretrained("carolanderson/roberta-base-food-ner", add_prefix_space=True)
# ner_model = AutoModelForTokenClassification.from_pretrained("carolanderson/roberta-base-food-ner")
# ner_pipe = pipeline("ner", model=ner_model, tokenizer=ner_tokenizer, aggregation_strategy="first", device=0)
processed_valid = [str(t) if pd.notna(t) else "" for t in df_food['clean_combined_text'].tolist()]
print(f"Total posts: {len(all_texts_df)} | Food posts: {len(df_food)}")
print(f"Date range: {df_food['created_date'].min()} to {df_food['created_date'].max()}")

# ── Embeddings ──────────────────────────────────────────────────────
try:
    print("Loading all-mpnet-base-v2...")
    st_model = SentenceTransformer("all-mpnet-base-v2")
    embeddings = st_model.encode( processed_valid, show_progress_bar=True, batch_size=64)
    print(f" Embeddings shape: {embeddings.shape}")
except Exception as e:
    print(f" mpnet failed ({e}), falling back to MiniLM")
    st_model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = st_model.encode(processed_valid, show_progress_bar=True,
                                  batch_size=64, normalize_embeddings=True)

# ── Dynamic clustering parameters ───────────────────────────────────
n_docs = len(processed_valid)
n_neighbors = min(10, max(8, n_docs // 15))
n_components = min(5, max(5, n_docs // 50))
min_cluster_size = 10
min_samples = 3

print(f"UMAP: n_neighbors={n_neighbors}, n_components={n_components}")
print(f"HDBSCAN: min_cluster_size={min_cluster_size}, min_samples={min_samples}")

umap_model = UMAP(
    n_neighbors=n_neighbors, n_components=n_components,
    spread=1.0, min_dist=0.0, metric="cosine", random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=min_cluster_size, min_samples=min_samples,
    metric="euclidean", cluster_selection_method="leaf", prediction_data=True
)

custom_stops = list(stopwords.words('english')) + [
    'food','eat','eating','like','just','got','really','one','get','go','going',
    'know','think','today','dublin','ireland','cork','galway','irish','new',
    'good','great','best','love','try','tried','make','made','time','day','week',
    'amazing','always','say','said','way','also','still','much','well','back',
    'want','need','see','look','looking','quot', 'amp', 'https', 'http', 'com', 'video', 'tiktok', 'shorts', 'fyp',
]

# vectorizer_model = CountVectorizer(
#     stop_words=CUSTOM_STOP_WORDS, ngram_range=(1, 3),
#     min_df=3, max_df=0.5, max_features=10000
# )

vectorizer_model = CountVectorizer(
    stop_words=custom_stops,
    ngram_range=(1, 3),
    min_df=2,           # low threshold for small dataset
    max_df=0.9
)

# ctfidf_model = ClassTfidfTransformer(
#     reduce_frequent_words=True, bm25_weighting=True, seed_words=SEED_WORDS
# )

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

representation_layers = [
    MaximalMarginalRelevance(diversity=0.5, top_n_words=15),
    KeyBERTInspired(top_n_words=15),
    PartOfSpeech("en_core_web_sm", top_n_words=15),
]

# ── Fit BERTopic ────────────────────────────────────────────────────
try:
    topic_model = BERTopic(
        embedding_model=st_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        representation_model=representation_layers,
        calculate_probabilities=True,
        verbose=True
    )
    
    topics, probs = topic_model.fit_transform(processed_valid, embeddings)
    
    #Outlier reduction
    if hasattr(topic_model, "reduce_outliers"):
        topics = topic_model.reduce_outliers(
            processed_valid, topics, strategy="c-tf-idf", threshold=0.1
        )
        topic_model.update_topics(
            processed_valid, topics=topics,
            vectorizer_model=vectorizer_model,
            ctfidf_model=ctfidf_model,
            representation_model=representation_layers
        )
    new_topics = topic_model.reduce_outliers(
    processed_valid,
    topics,
    strategy="embeddings",
    embeddings=embeddings
    )
    topic_model.update_topics(
        processed_valid,
        topics=new_topics,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model
    )
    
    all_texts_df = df_food.copy()
    all_texts_df["topic_id"] = new_topics
    
    topic_info = topic_model.get_topic_info()
    print(f"\nDiscovered {len(topic_info)-1} topics")
    display(topic_model.get_topic_info().head(15))

except Exception as e:
    print(f" BERTopic failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 6 — FOOD TOPIC VALIDATION (single API call, no caching, no chunking)
#
# Sends: topic_id + keywords only.  Gets back: valid flag + food name + category.
# Prints the first 10 inputs before calling the API.
#
# Output: food_topics_df  ->  topic_id | keywords | is_valid_food_topic |
#                             food_name | food_category | reason
#         saved to food_topics.csv
# ══════════════════════════════════════════════════════════════════════
import json
import re
import time

import pandas as pd
import litellm

# ── CONFIG ────────────────────────────────────────────────────────────
print(LLM_API_KEY)
print(LLM_MODEL)
print(LLM_PROVIDER)
# LLM_API_KEY comes from your dotenv cell

FOOD_RATIO_THRESHOLD = 0.8     # >=80% of keywords must be food related
REQUIRE_COHERENCE    = True    # keywords must form ONE food category
MAX_OUTPUT_TOKENS    = 16000

# ══════════════════════════════════════════════════════════════════════
# 1. BUILD INPUT — straight from topic_model, id + keywords only
# ══════════════════════════════════════════════════════════════════════
def _as_keyword_list(rep):
    if isinstance(rep, (list, tuple)):
        return [str(x) for x in rep]
    if rep is None:
        return []
    text = str(rep).strip()
    if text.startswith("["):
        try:
            return [str(x) for x in json.loads(text.replace("'", '"'))]
        except Exception:
            pass
    return [w.strip() for w in text.split(",") if w.strip()]


info = topic_model.get_topic_info()
info = info[info["Topic"] != -1]

topics = [
    {"id": int(r["Topic"]), "keywords": _as_keyword_list(r["Representation"])}
    for _, r in info.iterrows()
]

if not topics:
    raise RuntimeError(
        "No topics found in topic_model.get_topic_info(). "
        "Refit BERTopic — the model has no topics besides the -1 outlier."
    )

print(f"Topics to validate: {len(topics)}")
print("\n─── FIRST 10 INPUTS SENT TO THE LLM " + "─" * 26)
for t in topics[:10]:
    print(f"  {t['id']:>3} | {', '.join(t['keywords'])}")

print("\n─── last 10 INPUTS SENT TO THE LLM " + "─" * 26)
for t in topics[-10:]:
    print(f"  {t['id']:>3} | {', '.join(t['keywords'])}")
    
print("─" * 62)

payload = json.dumps(topics, ensure_ascii=False)
print(f"\nPayload size: ~{len(payload) // 4} tokens\n")


# ══════════════════════════════════════════════════════════════════════
# 2. THE PROMPT
# ══════════════════════════════════════════════════════════════════════
SYSTEM_PROMPT = """You validate BERTopic clusters from social-media food content (Ireland and elsewhere).

Each topic has ~10 keywords from c-TF-IDF. For each topic:

STEP 1 - Count how many keywords are FOOD related.
  FOOD     = a food, drink, ingredient, dish, cuisine; a food venue (chipper, deli, takeaway, pub, cafe);
             a meal occasion (breakfast, brunch, dinner); a cooking/eating action (recipe, cooking, ate,
             mukbang, tsp); a diet (vegan, keto, high protein); a food hashtag (foodtok, irishfood,
             whatieatinaday, veganfood, farmersmarket, streetfood).
             A multi-word keyword counts as FOOD if it contains a food word.
  NON_FOOD = places and countries; people and presenter names; platform tokens (viral, trending, fyp,
             fypシ, ytshorts, reels, pov, vlog, thank, wow); filler (best, worst, actually, must,
             legendary, average, massive); bare numbers; anything from an unrelated domain.

STEP 2 - Decide coherence. The food keywords must describe ONE food, dish, cuisine, meal occasion,
  diet or venue type. Overlapping n-grams of the same phrase count as ONE concept.
    coherent:     brownie + chocolate + cookies + dessert
    coherent:     trout + potatoes + beets + broccoli + steamed
    NOT coherent: tea + jam + currants + caramelized      (different categories)
    NOT coherent: brekkie + nutella + spice bag           (different categories)

TRAPS - apply these:
  - needoh / squishy / kawaii beside "happy meal" is TOY unboxing, not food.
  - graham norton, faze rug, gordon ramsay are people, not food.
  - visitmalaysia2026, blarney castle, temple bar, howth, nyc are places, not food.
  - celiac, famine, pesticides, poverty, food-safety rants are talk ABOUT food, not a food topic.
  - supervalu, tesco, aldi, cost, prices, haul are shopping topics, not food topics.
  - drinks (wine, whiskey, coffee, matcha, pints) DO count as food.

STEP 3 - Name it. food_name: 2-6 words, title case, naming the FOOD (e.g. "Homemade Rajbhog Ice Cream",
  "Vegan Porridge & Oatmeal Breakfast"). Never generic words like "Food", "Recipe", "Comfort",
  "Afternoon". If there is no food anchor, use "".

  food_category: pick the MOST SPECIFIC category that fits, not the broadest one. Use one of:
  Burgers & Grilled Meat, Fried Chicken, Chips & Fries, Pizza, Sandwiches & Wraps,
  Bakery & Bread, Cakes & Pastries, Cookies & Biscuits, Ice Cream & Frozen Desserts,
  Chocolate & Confectionery, Coffee & Tea, Alcoholic Drinks, Smoothies & Juices,
  Vegetarian & Vegan, Indian Cuisine, Chinese & Asian Takeaway, Irish Traditional,
  Breakfast & Brunch, Salad & Healthy Eating, Seafood, Stews & Comfort Food,
  Curry & Spiced Dishes, Snacks & Street Food, Meal Prep & Diet.
  Empty string if not a food topic.

  RULES:
  - Do NOT default to "Snacks", "Other Food", or a broad catch-all if a more specific
    category applies. E.g. a burger is "Burgers & Grilled Meat", not "Fast Food" or "Snacks".
  - Only use a broad category when the food genuinely doesn't fit any specific one above.
  - If you find yourself using the same category for more than 3-4 clearly different dishes,
    reconsider — you're probably being too broad.


Return a JSON array, one object per input topic, in the same order. No prose, no markdown fences:

[{"id": 0, "food_kw": 10, "total_kw": 10, "coherent": true,
  "food_name": "Vegan Plant-Based Daily Meals", "food_category": "Vegetarian & Vegan",
  "reason": "all keywords are vegan food hashtags"}]

food_kw = number of FOOD keywords. total_kw = total keywords given. reason = max 15 words.
Do NOT output a valid flag — the caller computes it."""

# SYSTEM_PROMPT = """You validate BERTopic clusters from social-media food content (Ireland and elsewhere).

# You will receive MULTIPLE topics in one request. Treat EVERY topic completely independently —
# do not let one topic's food_category influence another's, and do not batch similar-sounding
# topics into the same category just because they share a "street food" or "snack" hashtag.
# Each topic must be judged only on its own keywords.

# Each topic has ~10 keywords from c-TF-IDF. For each topic:

# STEP 1 - Count how many keywords are FOOD related.
#   FOOD     = a food, drink, ingredient, dish, cuisine; a food venue (chipper, deli, takeaway, pub, cafe);
#              a meal occasion (breakfast, brunch, dinner); a cooking/eating action (recipe, cooking, ate,
#              mukbang, tsp); a diet (vegan, keto, high protein); a food hashtag (foodtok, irishfood,
#              whatieatinaday, veganfood, farmersmarket, streetfood).
#              A multi-word keyword counts as FOOD if it contains a food word.
#   NON_FOOD = places and countries; people and presenter names; platform tokens (viral, trending, fyp,
#              fypシ, ytshorts, reels, pov, vlog, thank, wow); filler (best, worst, actually, must,
#              legendary, average, massive); bare numbers; anything from an unrelated domain.

# STEP 2 - Decide coherence. The food keywords must describe ONE food, dish, cuisine, meal occasion,
#   diet or venue type. Overlapping n-grams of the same phrase count as ONE concept.
#     coherent:     brownie + chocolate + cookies + dessert
#     NOT coherent: tea + jam + currants + caramelized      (different categories)

# TRAPS - apply these:
#   - needoh / squishy / kawaii beside "happy meal" is TOY unboxing, not food.
#   - graham norton, faze rug, gordon ramsay are people, not food.
#   - visitmalaysia2026, blarney castle, temple bar, howth, nyc are places, not food.
#   - celiac, famine, pesticides, poverty, food-safety rants are talk ABOUT food, not a food topic.
#   - supervalu, tesco, aldi, cost, prices, haul are shopping topics, not food topics.
#   - drinks (wine, whiskey, coffee, matcha, pints) DO count as food.
#   - "viral", "street food", "mukbang", "farmers market", "snacks" are FORMAT/VENUE words, never
#     the category itself — always look past them for the actual dish keyword underneath.

# STEP 3 - Name it. food_name: 2-6 words, title case, naming the FOOD. Never generic words like
#   "Food", "Recipe", "Comfort", "Afternoon". If there is no food anchor, use "".

# STEP 4 - Categorize. Find the actual dish/ingredient keyword first, then map it using this lookup.
#   A dish anchor always overrides "viral/snack/street food/farmers market/mukbang" wrapper words.
#     potato, fries, chips, wedges                    -> Chips & Fries
#     samosa, curry, masala, tikka, biryani            -> Indian Cuisine (or Curry & Spiced Dishes)
#     spice bag, boxty, coddle, full irish             -> Irish Traditional
#     burger, kofta, kebab, patty, grilled meat        -> Burgers & Grilled Meat
#     fried chicken, wings, nuggets                    -> Fried Chicken
#     duck, chicken, pork, beef (no dish name given)   -> Stews & Comfort Food
#     nachos, tacos, burrito, quesadilla               -> Snacks & Street Food
#     pizza                                            -> Pizza
#     cake, pastry, croissant, brownie                 -> Cakes & Pastries
#     cookie, biscuit                                  -> Cookies & Biscuits
#     ice cream, gelato                                -> Ice Cream & Frozen Desserts
#     chocolate, candy, sweets                         -> Chocolate & Confectionery
#     coffee, tea, latte, matcha                       -> Coffee & Tea
#     wine, beer, whiskey, pint, cocktail              -> Alcoholic Drinks
#     smoothie, juice                                  -> Smoothies & Juices
#     seafood, fish, prawns, salmon                    -> Seafood
#     sandwich, wrap, sub                               -> Sandwiches & Wraps
#     bread, bakery, sourdough                         -> Bakery & Bread
#     salad, healthy bowl                              -> Salad & Healthy Eating

#   Only if NO dish/ingredient anchor exists at all — keywords are purely diet-style (vegan, keto,
#   meal plan) -> Vegetarian & Vegan or Meal Prep & Diet. Purely mixed grazing with zero identifiable
#   dish -> Snacks & Street Food. Purely a meal-time reference with no dish named -> Breakfast & Brunch.

#   food_category must be copied verbatim from this full list, exact spelling, nothing invented:
#   Burgers & Grilled Meat, Fried Chicken, Chips & Fries, Pizza, Sandwiches & Wraps,
#   Bakery & Bread, Cakes & Pastries, Cookies & Biscuits, Ice Cream & Frozen Desserts,
#   Chocolate & Confectionery, Coffee & Tea, Alcoholic Drinks, Smoothies & Juices,
#   Vegetarian & Vegan, Indian Cuisine, Chinese & Asian Takeaway, Irish Traditional,
#   Breakfast & Brunch, Salad & Healthy Eating, Seafood, Stews & Comfort Food,
#   Curry & Spiced Dishes, Snacks & Street Food, Meal Prep & Diet.
#   Empty string if not a food topic.

#   RULES:
#   - Do NOT default to "Snacks & Street Food", "Vegetarian & Vegan", "Meal Prep & Diet", or
#     "Breakfast & Brunch" if a dish/ingredient anchor from the lookup is present.
#   - Before finalizing each topic, re-check its food_category against the lookup independently
#     of every other topic in this batch.
#   - If more than 3-4 topics in this batch end up with the same category, stop and recheck each
#     one individually against the lookup — you are likely defaulting instead of matching.

# STEP 5 - Flag generalization. Set too_generalized: true if food_category is one of these fallback
#   buckets — Snacks & Street Food, Vegetarian & Vegan, Meal Prep & Diet, Breakfast & Brunch,
#   Salad & Healthy Eating, Stews & Comfort Food, Curry & Spiced Dishes — AND a more specific
#   dish/ingredient anchor from STEP 4's lookup was present but not used, or no anchor could be
#   found at all. Set too_generalized: false if food_category is any other, more specific category,
#   OR if it is a fallback bucket used correctly (no dish anchor genuinely existed).
#   This flag lets the caller filter out weak/lazy category assignments downstream.

# Return a JSON array, one object per input topic, in the same order. No prose, no markdown fences:

# [{"id": 0, "food_kw": 10, "total_kw": 10, "coherent": true,
#   "food_name": "Vegan Plant-Based Daily Meals", "food_category": "Vegetarian & Vegan",
#   "too_generalized": false,
#   "reason": "all keywords are vegan food hashtags, no specific dish present"}]

# food_kw = number of FOOD keywords. total_kw = total keywords given. reason = max 15 words.
# """

USER_PROMPT = f"Validate these {len(topics)} topics:\n\n{payload}"


# ══════════════════════════════════════════════════════════════════════
# 3. SINGLE API CALL
#    No response_format: Groq's JSON mode returns json_validate_failed on
#    reasoning models. We parse the array out of the text instead.
# ══════════════════════════════════════════════════════════════════════
def extract_json_array(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)       # reasoning traces
    text = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", text.strip())     # fences
    start, end = text.find("["), text.rfind("]")
    if start == -1 or end == -1:
        raise ValueError(f"No JSON array in response. First 300 chars:\n{text[:300]}")
    return json.loads(text[start:end + 1])


print("Calling LLM (1 request)...")
for attempt in range(1):
    try:
        response = litellm.completion(
            model=f"{LLM_PROVIDER}/{LLM_MODEL}",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_PROMPT},
            ],
            api_key=LLM_API_KEY,
            temperature=0,
            max_tokens=MAX_OUTPUT_TOKENS,
        )
        
        break
    except Exception as e:
        print(f"  ERROR TYPE: {type(e).__name__}")
        print(f"  ERROR MSG: {str(e)}")
        if "rate" in str(e).lower() and attempt < 2:
            wait = 20 * (attempt + 1)
            print(f"  rate limited — waiting {wait}s")
            time.sleep(wait)
            continue
        raise

raw_LLM_output = response["choices"][0]["message"]["content"]
LLM_output_results = extract_json_array(raw_LLM_output)
print(f"Got {len(LLM_output_results)} results back for {len(LLM_output_results)} topics\n")
LLM_output_DataFrames = pd.DataFrame(LLM_output_results)
LLM_output_DataFrames_ValidFoods = LLM_output_DataFrames[LLM_output_DataFrames['coherent'] == True]
#LLM_output_DataFrames_ValidFoods = LLM_output_DataFrames[ (LLM_output_DataFrames['coherent'] == True) & (LLM_output_DataFrames['too_generalized'] == False)]
grouped_LLM_output_DataFrames = (
    LLM_output_DataFrames_ValidFoods.groupby('food_category')
     .agg(
         topic_ids=('id', lambda x: ', '.join(map(str, sorted(x)))),
         food_names=('food_name', lambda x: ', '.join(sorted(set(x)))),
         topic_count=('id', 'count'),
         total_food_kw=('food_kw', 'sum'),
     )
     .reset_index()
     .sort_values('topic_count', ascending=False)
)

display(grouped_LLM_output_DataFrames)


In [ ]:
grouped_LLM_output_DataFrames.to_excel('t.xlsx')

In [ ]:
# Keep only valid/coherent food topics
topic_food_mapping_df = (
    LLM_output_DataFrames[
        LLM_output_DataFrames["coherent"].eq(True)
    ][["id", "food_name", "food_category"]]
    .rename(columns={"id": "topic_id"})
    .sort_values("topic_id")
    .reset_index(drop=True)
)

display(topic_food_mapping_df)

print(f"Valid food topics: {len(topic_food_mapping_df)} / {len(LLM_output_DataFrames)}")

In [ ]:
display(all_texts_df.head(5))

topic_food_mapping_df.to_excel("foodcategories.xlsx")

In [ ]:
# Ensure topic_id has the same type in both DataFrames
all_texts_df["topic_id"] = pd.to_numeric(
    all_texts_df["topic_id"],
    errors="coerce"
).astype("Int64")

topic_food_mapping_df["topic_id"] = pd.to_numeric(
    topic_food_mapping_df["topic_id"],
    errors="coerce"
).astype("Int64")

# Remove existing mapping columns before re-running the cell
all_texts_df = all_texts_df.drop(
    columns=["food_category"],
    errors="ignore"
)

# Add topic label and food category
all_texts_df = all_texts_df.merge(
    topic_food_mapping_df[
        ["topic_id", "food_category"]
    ],
    on="topic_id",
    how="left",
    validate="many_to_one"
)


if DB_AVAILABLE:
    try:
        id_df = pd.read_sql(
            """
            SELECT
                id AS raw_post_id,
                platform,
                source,
                title,
                created_date::date AS created_date_key
            FROM raw_posts
            """,
            engine
        )

        # Convert both join columns to identical Python date values
        id_df["created_date_key"] = pd.to_datetime(
            id_df["created_date_key"],
            errors="coerce",
            utc=True
        ).dt.date

        all_texts_df["created_date_key"] = pd.to_datetime(
            all_texts_df["created_date"],
            errors="coerce",
            utc=True
        ).dt.date

        # Remove raw_post_id before rerunning this cell
        all_texts_df = all_texts_df.drop(
            columns=["raw_post_id"],
            errors="ignore"
        )

        joined_ids = (
            id_df[
                [
                    "raw_post_id",
                    "platform",
                    "source",
                    "title",
                    "created_date_key"
                ]
            ]
            .drop_duplicates(
                subset=[
                    "platform",
                    "source",
                    "title",
                    "created_date_key"
                ]
            )
        )

        all_texts_df = all_texts_df.merge(
            joined_ids,
            on=[
                "platform",
                "source",
                "title",
                "created_date_key"
            ],
            how="left",
            validate="many_to_one"
        )

        all_texts_df = all_texts_df.drop(
            columns=["created_date_key"]
        )

        matched = all_texts_df["raw_post_id"].notna().sum()

        print(
            f"raw_post_id joined: "
            f"{matched}/{len(all_texts_df)} rows matched"
        )

    except Exception as e:
        print(f"Warning: could not join raw_post_id: {e}")

        all_texts_df = all_texts_df.drop(
            columns=["created_date_key"],
            errors="ignore"
        )

        if "raw_post_id" not in all_texts_df.columns:
            all_texts_df["raw_post_id"] = pd.NA

else:
    all_texts_df["raw_post_id"] = pd.NA

In [ ]:
# Prepare only posts that have a valid food topic mapping
posts_with_topics_df = (
    all_texts_df[
        all_texts_df["food_category"].notna()
        & all_texts_df["topic_id"].notna()
    ]
    .copy()
)

# raw_posts.id becomes the foreign key
posts_with_topics_df["raw_post_id"] = pd.to_numeric(
    posts_with_topics_df["id"],
    errors="coerce"
).astype("Int64")

# Remove rows without a valid raw_posts ID
posts_with_topics_df = posts_with_topics_df[
    posts_with_topics_df["raw_post_id"].notna()
].copy()

# Select only columns present in the PostTopic model
posts_with_topics_df = posts_with_topics_df[
    [
        "raw_post_id",
        "platform",
        "source",
        "title",
        "full_text",
        "week_start_date",
        "topic_id",
        "food_category",
    ]
]

# Ensure appropriate types
posts_with_topics_df["raw_post_id"] = (
    posts_with_topics_df["raw_post_id"].astype(int)
)

posts_with_topics_df["topic_id"] = (
    pd.to_numeric(
        posts_with_topics_df["topic_id"],
        errors="coerce"
    )
    .astype(int)
)

print(f"Rows ready to save: {len(posts_with_topics_df)}")

display(posts_with_topics_df.head())

In [ ]:
# !Important Don't run this code multiple Time
# If you want to run this again run the below code first
all_texts_df.drop(
 columns=["food_category"],
 inplace=True,
 errors="ignore"
)

# 1. Build topic_id -> food_category lookup from the grouped dataframe
lookup = (
    grouped_LLM_output_DataFrames[['food_category', 'topic_ids']]
    .assign(topic_id=lambda d: d['topic_ids'].astype(str).str.split(','))
    .explode('topic_id')
)
lookup['topic_id'] = pd.to_numeric(lookup['topic_id'].str.strip(), errors='coerce')
lookup = lookup.dropna(subset=['topic_id'])
lookup['topic_id'] = lookup['topic_id'].astype(int)
lookup = lookup[['topic_id', 'food_category']].drop_duplicates(subset='topic_id')

# 2. Merge onto all_texts_df
all_texts_df['topic_id'] = pd.to_numeric(all_texts_df['topic_id'], errors='coerce').astype('Int64')

all_texts_df = all_texts_df.merge(lookup, on='topic_id', how='left')

# 3. Anything not in a valid food topic (outliers = -1, or topics that failed validation)
all_texts_df['food_category'] = all_texts_df['food_category'].fillna('Unassigned')
display(all_texts_df[all_texts_df['food_category'] != 'Unassigned'])


In [ ]:
write_to_db(posts_with_topics_df,"posts_with_topics", if_exists="replace")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 8: WEEKLY FEATURE ENGINEERING ( FINAL FEATURES + LABEL)
# ══════════════════════════════════════════════════════════════════════

# ── Step 1: Aggregate weekly by category ────────────────────────────
weekly_volume = (
    all_texts_df[all_texts_df["food_category"] != "Unassigned"]
    .groupby(["food_category", "week_start_date"])
    .agg(
        total_posts=("full_text", "count"),
        platform_count=("platform", "nunique"),
    )
    .reset_index()
)


# ── Step 2: Normalise the join key ─────────────────────────────────
# The merge in Step 4 matches on EXACT values, so the key must be a plain
# midnight, tz-naive Timestamp. Without this, "2026-01-19" (str) or
# 2026-01-19 18:30:00 won't equal Timestamp("2026-01-19") in the grid and
# the merge silently returns all-NaN — which reads as "no data" not "bug".
#   utc=True        -> parses strings and unifies mixed tz-aware/naive input
#   tz_convert(None)-> drops the tz so it can compare against a naive grid
#   normalize()     -> strips the time component down to 00:00:00
weekly_volume["week_start_date"] = (
    pd.to_datetime(weekly_volume["week_start_date"], utc=True)
      .dt.tz_convert(None)
      .dt.normalize()
)

# .dt.weekday gives 0 for Monday, 1 for Tuesday, ... 6 for Sunday.
# Subtracting that many days from the date forces it back to the Monday of that week, no matter what day of the week the date currently is.
weekly_volume["week_start_date"] = (
    weekly_volume["week_start_date"]
    - pd.to_timedelta(weekly_volume["week_start_date"].dt.weekday, unit="D")
)

print(len(weekly_volume))

# ── Step 3: Build full grid (category × all weeks) ─────────────────
# groupby can only emit rows for data that EXISTS, so a category with zero
# posts in a given week has no row at all. That makes a drop-to-zero look
# like a straight line between the two weeks that do have data, and throws
# off any rolling window. Fix: build every (category, week) pair up front.
#   freq="W-MON" -> weekly, anchored on Monday. Safe only because Step 2
#                   guarantees min() already sits on a Monday; an anchored
#                   offset rolls forward otherwise and shifts the whole grid.
all_weeks = pd.date_range(
    weekly_volume["week_start_date"].min(),
    weekly_volume["week_start_date"].max(),
    freq="W-MON"
)

all_categories = weekly_volume["food_category"].unique()

full_grid = pd.MultiIndex.from_product(
    [all_categories, all_weeks],
    names=["food_category", "week_start_date"]
).to_frame(index=False)

# ── Step 3: Build full grid (category × all weeks) ─────────────────
# groupby can only emit rows for data that EXISTS, so a category with zero
# posts in a given week has no row at all. That makes a drop-to-zero look
# like a straight line between the two weeks that do have data, and throws
# off any rolling window. Fix: build every (category, week) pair up front.
#   freq="W-MON" -> weekly, anchored on Monday. Safe only because Step 2
#                   guarantees min() already sits on a Monday; an anchored
#                   offset rolls forward otherwise and shifts the whole grid.
weekly_volume = full_grid.merge(
    weekly_volume,
    on=["food_category", "week_start_date"],
    how="left"
).sort_values(["food_category", "week_start_date"]).reset_index(drop=True)


# # ══════════════════════════════════════════════════════════════════════
# # FOUNDATION Feature: Total Post
# # total_posts[t] = count(posts in week t) 
# # ══════════════════════════════════════════════════════════════════════

# # Fill missing values ─────────────────────────────────────
weekly_volume["total_posts"] = weekly_volume["total_posts"].fillna(0)

# ══════════════════════════════════════════════════════════════════════
# FEATURE 1: PlatForm Count
# platform_count[t] = nunique(platforms in week t) 
# ══════════════════════════════════════════════════════════════════════
weekly_volume["platform_count"] = weekly_volume["platform_count"].fillna(0)


# ══════════════════════════════════════════════════════════════════════
# FEATURE 2: Weightaged Rolling Average 
# rolling_avg[t] = 0.6·posts[t] + 0.3·posts[t-1] + 0.1·posts[t-2] 
# ══════════════════════════════════════════════════════════════════════

g  = weekly_volume.groupby("food_category")["total_posts"]
P  = weekly_volume["total_posts"]
P1 = g.shift(1)
P2 = g.shift(2)

weekly_volume["rolling_avg"] = 0.6*P + 0.3*P1.fillna(0) + 0.1*P2.fillna(0)


# ══════════════════════════════════════════════════════════════════════
# FEATURE 3: Ratio to Peak — position relative to its own ceiling
# ratio_to_peak[t] = posts[t] / max(posts[1...t])
# ══════════════════════════════════════════════════════════════════════

# expanding().max() = running maximum from the first week up to and
# including t. No leakage — it never sees a future week.
peak_so_far = weekly_volume.groupby("food_category")["total_posts"].transform(
    lambda x: x.expanding().max()
)

# Guard the divide: categories that are still all-zero have peak == 0,
# and 0/0 -> NaN. Those weeks get 0.0 (no activity, no position).
weekly_volume["ratio_to_peak"] = np.where(
    peak_so_far > 0,
    weekly_volume["total_posts"] / peak_so_far,
    0.0
)

# ══════════════════════════════════════════════════════════════════════
# FEATURE 4: Sustained Growth — the fad-filter
# If posts[t] > posts[t-1]: streak+1, else reset to 0
# ══════════════════════════════════════════════════════════════════════

# Did this week beat last week? shift(1) is per-category, so the streak
# never carries across a category boundary. Week 1 has no prior week ->
# NaN comparison -> False -> streak starts at 0.
is_up = (
    weekly_volume["total_posts"]
    > weekly_volume.groupby("food_category")["total_posts"].shift(1)
).astype(int)

# The reset trick: (is_up == 0).cumsum() ticks up by 1 at every break,
# so each unbroken run of up-weeks shares a unique group id. cumsum
# within that group counts the streak, and it restarts automatically
# at the next break — no loop needed.
weekly_volume["sustained_growth"] = is_up.groupby(
    [weekly_volume["food_category"], (is_up == 0).cumsum()]
).cumsum()


# ══════════════════════════════════════════════════════════════════════
# FEATURE 5: Growth Rate (momentum)
# growth_rate[t] = (rolling_avg[t] − rolling_avg[t-1]) / rolling_avg[t-1] (else 0.0) 
# ══════════════════════════════════════════════════════════════════════

# Previous week's smoothed value. shift(1) is per-category, so week 1 of
# a category never borrows the last week of the one before it.
prev_avg = weekly_volume.groupby("food_category")["rolling_avg"].shift(1)

# Guard the divide. After the grid fill, rolling_avg[t-1] == 0 is common
# (every quiet stretch), and (x - 0)/0 -> inf, which silently poisons any
# scaling or model fit. Zero-baseline weeks get 0.0 instead.
# NaN prev_avg (week 1, plus the min_periods=3 warm-up) also falls to 0.0.
weekly_volume["growth_rate"] = np.where(
    prev_avg > 0,
    (weekly_volume["rolling_avg"] - prev_avg) / prev_avg,
    0.0
)



# ══════════════════════════════════════════════════════════════════════
# LABEL INPUT: future_avg — activity over the NEXT 2 weeks
# future_avg[t] = (posts[t+1] + posts[t+2]) / 2
# ⚠ Uses future rows. For building the label ONLY — never feed as a feature.
# ══════════════════════════════════════════════════════════════════════
weekly_volume["future_avg"] = weekly_volume.groupby("food_category")["total_posts"].transform(
    lambda x: x.shift(-2).rolling(window=2, min_periods=2).mean()
)


# ══════════════════════════════════════════════════════════════════════
# FEATURE 6: ra4
# ra4[t] = mean(posts[t-3..t]) 
# ══════════════════════════════════════════════════════════════════════
weekly_volume["ra4"] = weekly_volume.groupby("food_category")["total_posts"].transform(
    lambda x: x.rolling(4, min_periods=1).mean()
)

# ══════════════════════════════════════════════════════════════════════
# FEATURE 7: rank_wk
# rank_wk[t] = pct_rank(rolling_avg[t] within week t) 
# ═════════════════════════════════════════════════════════════════════
weekly_volume["rank_wk"] = weekly_volume.groupby("week_start_date")["rolling_avg"].rank(pct=True)

print("Total rows in weekly_volume:", len(weekly_volume))

model_df = weekly_volume[weekly_volume["future_avg"].notna()].copy()

print("Rows after removing NaN future_avg:", len(model_df))
print("Rows removed:", len(weekly_volume) - len(model_df))

# ══════════════════════════════════════════════════════════════════════
# LABEL INPUT: will_trend
# will_trend[t] = 1 if future_avg[t] ≥ quantile₈₀(future_avg[·, week=t]) else 0 
# ══════════════════════════════════════════════════════════════════════

model_df["will_trend"] = (
    model_df["future_avg"] >= model_df.groupby("week_start_date")["future_avg"]
                                      .transform(lambda x: x.quantile(0.80))
).astype(int)

model_df = model_df.sort_values(
    ["week_start_date", "food_category"]
).reset_index(drop=True)

model_df = (
    model_df
    .sort_values(
        ["food_category", "week_start_date"]
    )
    .reset_index(drop=True)
)

display(model_df[model_df['will_trend'] == 1])


In [ ]:
display(model_df)

print(len(model_df))


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL Extra validation and Db conversion
# Features:
#   1. platform_count
#   2. rolling_avg
#   3. ratio_to_peak
#   4. sustained_growth
#   5. growth_rate
#   6. ra4
#   7. rank_wk
#
# Label inputs:
#   future_avg
#   will_trend
# ══════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd


# ══════════════════════════════════════════════════════════════════════
# STEP 0: VALIDATE REQUIRED COLUMNS
# ══════════════════════════════════════════════════════════════════════

required_columns = [
    "food_category",
    "week_start_date",
    "full_text",
    "platform",
]

missing_columns = [
    col for col in required_columns
    if col not in all_texts_df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns in all_texts_df: {missing_columns}"
    )

model_df["will_trend"] = model_df["will_trend"].astype(bool)

display(model_df.head(10))

# ══════════════════════════════════════════════════════════════════════
# FINAL COLUMN ORDER AND SORTING
# ══════════════════════════════════════════════════════════════════════

model_columns = [
    "food_category",
    "week_start_date",

    # Foundation column
    "total_posts",

    # Model features
    "platform_count",
    "rolling_avg",
    "ratio_to_peak",
    "sustained_growth",
    "growth_rate",
    "ra4",
    "rank_wk",

    # Future-derived analysis/label columns
    "future_avg",

    # Target
    "will_trend",
]

print(len(model_df))

model_df = (model_df
            .sort_values("total_posts", ascending=False)
            .drop_duplicates(["food_category", "week_start_date"], keep="first"))

print(len(model_df))

model_df = (
    model_df[model_columns]
    .sort_values(
        [
            "week_start_date",
            "food_category",
        ]
    )
    .reset_index(drop=True)
)


In [ ]:
write_to_db(model_df, "weekly_snapshots", if_exists="replace")

In [ ]:
# Traning Code Primary
# ══════════════════════════════════════════════════════════════════════
# CELL 9: EXPANDING-WINDOW TRAINING & MODEL SELECTION
#
# Assumes `model_df` is already in memory with these columns:
#   food_category, week_start_date, total_posts, platform_count,
#   rolling_avg, ratio_to_peak, sustained_growth, growth_rate,
#   future_avg, forward_growth, will_trend
# ══════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score
)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    HAS_XGB = False
    print("xgboost not installed — using sklearn GradientBoosting as stand-in.")

RANDOM_STATE = 42

# The 5 features. future_avg and forward_growth are NOT here — they are
# built from future weeks and would hand the model the answer directly.
FEATURES = [
    "total_posts",        # this week's raw activity
    "rolling_avg",        # weighted 3-week size (0.6/0.3/0.1)
    "ra4",                # 4-week mean — steadier view of size
    "rank_wk",            # percentile among all categories THIS week
    "growth_rate",        # momentum
    "sustained_growth",   # consistency (fad filter)
    "platform_count",     # cross-platform spread
    "ratio_to_peak",      # position vs its own ceiling
]
TARGET = "will_trend"


# ══════════════════════════════════════════════════════════════════════
# PART 1 — Build the expanding-window folds
# ══════════════════════════════════════════════════════════════════════

def make_expanding_folds(weeks, initial_train=10, gap=2, test_size=2, step=2):
    """
    Round 1: train weeks 1-10  | skip 11-12 | test 13-14
    Round 2: train weeks 1-12  | skip 13-14 | test 15-16
    Round 3: train weeks 1-14  | skip 15-16 | test 17-18   ... and so on.

    The training block GROWS each round (that's "expanding"); the test
    block always sits AFTER it (never before — that would be predicting
    the past from the future).

    Why the gap: forward_growth at week t is built from weeks t+1 and
    t+2. Without a 2-week buffer, a training row at week t already
    contains the outcome of the first test week. The gap is what makes
    the score honest.
    """
    folds = []
    train_end = initial_train
    while train_end + gap + test_size <= len(weeks):
        train_weeks = weeks[:train_end]
        test_weeks = weeks[train_end + gap: train_end + gap + test_size]
        folds.append((train_weeks, test_weeks))
        train_end += step
    return folds


weeks = sorted(model_df["week_start_date"].unique())
folds = make_expanding_folds(weeks)

print(f"{len(weeks)} weeks available -> {len(folds)} rounds\n")
for i, (tr, te) in enumerate(folds, 1):
    print(f"  Round {i}: train {pd.Timestamp(tr[0]).date()} -> {pd.Timestamp(tr[-1]).date()}"
          f"  ({len(tr)} wks) | test {pd.Timestamp(te[0]).date()} -> {pd.Timestamp(te[-1]).date()}")


# ══════════════════════════════════════════════════════════════════════
# PART 2 — Define the 3 models
# ══════════════════════════════════════════════════════════════════════
# A fresh model is built from scratch every round. Nothing carries over
# between rounds — only the scores survive.
#
# class_weight="balanced" because only ~19% of rows are trends and
# catching real trends is the priority. Without it all three models
# drift toward predicting "not a trend" for everything.

def build_models():
    models = {
        "LogisticRegression": Pipeline([
            # scaling matters here only — rolling_avg spans 0-500 while
            # ratio_to_peak spans 0-1, and LR is sensitive to that.
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
            )),
        ]),
        "RandomForest": RandomForestClassifier(
            n_estimators=300, max_depth=4, min_samples_leaf=5,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
        ),
    }
    if HAS_XGB:
        models["XGBoost"] = XGBClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="logloss", random_state=RANDOM_STATE,
        )
    else:
        models["GradientBoosting"] = GradientBoostingClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05,
            random_state=RANDOM_STATE
        )
    return models


# ══════════════════════════════════════════════════════════════════════
# PART 3 — Run every model through every round
# ══════════════════════════════════════════════════════════════════════

fold_rows = []       # one row per (model, round)
conf_totals = {}     # model -> summed 2x2 confusion matrix

for name in build_models():
    conf_totals[name] = np.zeros((2, 2), dtype=int)

for r, (train_weeks, test_weeks) in enumerate(folds, 1):
    train = model_df[model_df["week_start_date"].isin(train_weeks)]
    test = model_df[model_df["week_start_date"].isin(test_weeks)]

    X_tr, y_tr = train[FEATURES], train[TARGET]
    X_te, y_te = test[FEATURES], test[TARGET]

    # A round with no positive test rows can't produce recall or AUC.
    # Record it as NaN rather than silently scoring 0.
    for name, model in build_models().items():

        # scale_pos_weight is XGBoost's version of class_weight="balanced"
        if name == "XGBoost":
            n_pos = max(int(y_tr.sum()), 1)
            model.set_params(scale_pos_weight=(len(y_tr) - n_pos) / n_pos)

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        y_prob = model.predict_proba(X_te)[:, 1]

        # labels=[0,1] forces a full 2x2 even if a class is absent
        cm = confusion_matrix(y_te, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        conf_totals[name] += cm

        both_classes = y_te.nunique() == 2

        fold_rows.append({
            "round": r,
            "model": name,
            "n_train": len(train), "n_test": len(test),
            "test_positives": int(y_te.sum()),
            "TP": tp, "FN": fn, "FP": fp, "TN": tn,
            "recall": recall_score(y_te, y_pred, zero_division=0),
            "precision": precision_score(y_te, y_pred, zero_division=0),
            "f1": f1_score(y_te, y_pred, zero_division=0),
            "auc": roc_auc_score(y_te, y_prob) if both_classes else np.nan,
        })

fold_scores = pd.DataFrame(fold_rows)

print("\n" + "=" * 78)
print("PER-ROUND SCORES")
print("=" * 78)
for name in fold_scores["model"].unique():
    sub = fold_scores[fold_scores["model"] == name]
    print(f"\n{name}")
    print(sub[["round", "n_train", "n_test", "test_positives",
               "TP", "FN", "FP", "TN",
               "recall", "precision", "f1", "auc"]].to_string(index=False))


# ══════════════════════════════════════════════════════════════════════
# PART 4 — Combine the rounds into one scorecard per model
# ══════════════════════════════════════════════════════════════════════
# mean  = the honest estimate of performance
# std   = stability. Similar scores round to round -> trustworthy.
#         Wild swings -> unstable model, treat the mean with suspicion.

METRICS = ["recall", "precision", "f1", "auc"]

summary = (
    fold_scores.groupby("model")[METRICS]
    .agg(["mean", "std"])
    .round(3)
)
summary.columns = [f"{m}_{s}" for m, s in summary.columns]

# Total caught vs missed across all rounds combined
totals = pd.DataFrame([
    {"model": n,
     "total_TP": cm[1, 1], "total_FN": cm[1, 0],
     "total_FP": cm[0, 1], "total_TN": cm[0, 0]}
    for n, cm in conf_totals.items()
]).set_index("model")

scorecard = summary.join(totals)

print("\n" + "=" * 78)
print("AVERAGED SCORECARD  (mean +/- std across rounds)")
print("=" * 78)
print(scorecard.to_string())

print("\nCombined confusion matrices (all rounds summed):")
for name, cm in conf_totals.items():
    print(f"\n  {name}")
    print(f"                   predicted_no   predicted_trend")
    print(f"    actual_no      {cm[0,0]:>12}   {cm[0,1]:>15}")
    print(f"    actual_trend   {cm[1,0]:>12}   {cm[1,1]:>15}   <- missed trends: {cm[1,0]}")


# ══════════════════════════════════════════════════════════════════════
# PART 5 — Pick the model to deploy
# ══════════════════════════════════════════════════════════════════════
# AUC ranks the models overall; recall says how many real trends were
# caught; F1 confirms the recall isn't faked by flagging everything.
#
# On small data these usually land close together. When they do, take
# the SIMPLEST (LogisticRegression) — least likely to overfit and you
# can read its coefficients. The boosted model is the benchmark that
# proves the simple one is good enough, not necessarily the winner.

TIE_TOLERANCE = 0.03      # AUC within this counts as "tied"
SIMPLICITY_ORDER = ["LogisticRegression", "RandomForest", "XGBoost", "GradientBoosting"]

best_auc = scorecard["auc_mean"].max()
tied = scorecard[scorecard["auc_mean"] >= best_auc - TIE_TOLERANCE].index.tolist()
chosen = min(tied, key=lambda m: SIMPLICITY_ORDER.index(m))

print("\n" + "=" * 78)
print(f"Best mean AUC: {best_auc:.3f}")
print(f"Within {TIE_TOLERANCE} of it: {tied}")
print(f"CHOSEN (simplest among the tied): {chosen}")
print("=" * 78)


# ══════════════════════════════════════════════════════════════════════
# PART 6 — Build the final deployed model
# ══════════════════════════════════════════════════════════════════════
# Train the chosen TYPE on ALL the data. The 5 round-models were
# throwaway probes used only to produce the scorecard; this is the one
# that goes live, and it gets every week of history.
#
# There is deliberately NO test here. It has seen all the data, so any
# score would be it grading its own homework. The Part 4 scorecard IS
# the performance estimate for this model — that's what the rounds were
# for.

final_model = build_models()[chosen]
if chosen == "XGBoost":
    n_pos = max(int(model_df[TARGET].sum()), 1)
    final_model.set_params(scale_pos_weight=(len(model_df) - n_pos) / n_pos)

final_model.fit(model_df[FEATURES], model_df[TARGET])

print(f"\nFinal model: {chosen} trained on all {len(model_df)} rows "
      f"({len(weeks)} weeks). Not scored — see the scorecard above.")

# What the model actually keyed on — sanity-check this against intuition
if chosen == "LogisticRegression":
    coefs = pd.Series(final_model.named_steps["clf"].coef_[0], index=FEATURES)
    print("\nCoefficients (positive = pushes toward 'will trend'):")
    print(coefs.sort_values(ascending=False).round(3).to_string())
elif hasattr(final_model, "feature_importances_"):
    imps = pd.Series(final_model.feature_importances_, index=FEATURES)
    print("\nFeature importances:")
    print(imps.sort_values(ascending=False).round(3).to_string())

# import joblib; joblib.dump(final_model, "trend_model.pkl")

In [ ]:
# Traning Code B
# ══════════════════════════════════════════════════════════════════════
# CELL 9: EXPANDING-WINDOW TRAINING & MODEL SELECTION
#
# Assumes `model_df` is already in memory with these columns:
#   food_category, week_start_date, total_posts, platform_count,
#   rolling_avg, ra4, rank_wk, ratio_to_peak, sustained_growth,
#   growth_rate, future_avg, will_trend
#
# will_trend is the leak-free label: 1 when future_avg is in the top 20%
# of that same week's future_avg values. No ratio, nothing from FEATURES
# inside the target.
# ══════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score
)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    HAS_XGB = False
    print("xgboost not installed — using sklearn GradientBoosting as stand-in.")

RANDOM_STATE = 42

# 8 features. future_avg is NOT here — it is built from future weeks and
# would hand the model the answer directly.
FEATURES = [
    "total_posts",        # this week's raw activity
    "rolling_avg",        # weighted 3-week size (0.6/0.3/0.1)
    "ra4",                # 4-week mean — steadier view of size
    "rank_wk",            # percentile among all categories THIS week
    "growth_rate",        # momentum
    "sustained_growth",   # consistency (fad filter)
    "platform_count",     # cross-platform spread
    "ratio_to_peak",      # position vs its own ceiling
]
TARGET = "will_trend"

# ── Operating point ────────────────────────────────────────────────
# Recall is the first priority, precision second. Rather than cutting at
# an arbitrary 0.50, take the HIGHEST cutoff that still holds recall at
# MIN_RECALL on the training rows. That buys precision out of the slack
# above the floor and never trades away recall you haven't allowed.
MIN_RECALL = 0.70


# ══════════════════════════════════════════════════════════════════════
# PART 1 — Build the expanding-window folds
# ══════════════════════════════════════════════════════════════════════

def make_expanding_folds(weeks, initial_train=10, gap=2, test_size=2, step=2):
    """
    Round 1: train weeks 1-10  | skip 11-12 | test 13-14
    Round 2: train weeks 1-12  | skip 13-14 | test 15-16
    Round 3: train weeks 1-14  | skip 15-16 | test 17-18   ... and so on.

    The training block GROWS each round (that's "expanding"); the test
    block always sits AFTER it (never before — that would be predicting
    the past from the future).

    Why the gap: future_avg at week t is built from weeks t+1 and t+2.
    Without a 2-week buffer, a training row at week t already contains
    the outcome of the first test week. The gap is what makes the score
    honest.
    """
    folds = []
    train_end = initial_train
    while train_end + gap + test_size <= len(weeks):
        train_weeks = weeks[:train_end]
        test_weeks = weeks[train_end + gap: train_end + gap + test_size]
        folds.append((train_weeks, test_weeks))
        train_end += step
    return folds


weeks = sorted(model_df["week_start_date"].unique())
folds = make_expanding_folds(weeks)

print(f"{len(weeks)} weeks available -> {len(folds)} rounds\n")
for i, (tr, te) in enumerate(folds, 1):
    print(f"  Round {i}: train {pd.Timestamp(tr[0]).date()} -> {pd.Timestamp(tr[-1]).date()}"
          f"  ({len(tr)} wks) | test {pd.Timestamp(te[0]).date()} -> {pd.Timestamp(te[-1]).date()}")


# ══════════════════════════════════════════════════════════════════════
# PART 2 — Define the 3 models
# ══════════════════════════════════════════════════════════════════════
# A fresh model is built from scratch every round. Nothing carries over
# between rounds — only the scores survive.
#
# class_weight="balanced" because only ~20% of rows are trends and
# catching real trends is the priority. Without it all three models
# drift toward predicting "not a trend" for everything.

def build_models():
    models = {
        "LogisticRegression": Pipeline([
            # scaling matters for LR only — rolling_avg spans 0-25 while
            # rank_wk and ratio_to_peak span 0-1
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
            )),
        ]),
        "RandomForest": RandomForestClassifier(
            n_estimators=300, max_depth=4, min_samples_leaf=5,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
        ),
    }
    if HAS_XGB:
        models["XGBoost"] = XGBClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="logloss", random_state=RANDOM_STATE,
        )
    else:
        models["GradientBoosting"] = GradientBoostingClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05,
            random_state=RANDOM_STATE
        )
    return models


def pick_threshold(y_true, probs, min_recall=MIN_RECALL):
    """
    Highest cutoff that still holds recall >= min_recall.
    Fitted on TRAIN rows only — fitting it on test would be reading the
    answers before the exam. Falls back to best-F1 if the floor is
    unreachable (can happen in an early round with few positives).
    """
    grid = np.linspace(0.20, 0.80, 61)
    ok = [t for t in grid
          if recall_score(y_true, (probs >= t).astype(int), zero_division=0) >= min_recall]
    if ok:
        return float(max(ok))
    return float(max(grid, key=lambda t: f1_score(y_true, (probs >= t).astype(int),
                                                  zero_division=0)))


# ══════════════════════════════════════════════════════════════════════
# PART 3 — Run every model through every round
# ══════════════════════════════════════════════════════════════════════

fold_rows = []                                        # one row per (model, round)
conf_totals = {n: np.zeros((2, 2), dtype=int) for n in build_models()}
thresholds = {n: [] for n in build_models()}
oof = {n: [] for n in build_models()}                 # pooled test predictions

for r, (train_weeks, test_weeks) in enumerate(folds, 1):
    train = model_df[model_df["week_start_date"].isin(train_weeks)]
    test = model_df[model_df["week_start_date"].isin(test_weeks)]

    X_tr, y_tr = train[FEATURES], train[TARGET]
    X_te, y_te = test[FEATURES], test[TARGET]

    for name, model in build_models().items():

        # scale_pos_weight is XGBoost's version of class_weight="balanced"
        if name == "XGBoost":
            n_pos = max(int(y_tr.sum()), 1)
            model.set_params(scale_pos_weight=(len(y_tr) - n_pos) / n_pos)

        model.fit(X_tr, y_tr)

        thr = pick_threshold(y_tr, model.predict_proba(X_tr)[:, 1])
        thresholds[name].append(thr)

        y_prob = model.predict_proba(X_te)[:, 1]
        y_pred = (y_prob >= thr).astype(int)          # replaces model.predict()

        # labels=[0,1] forces a full 2x2 even if a class is absent
        cm = confusion_matrix(y_te, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        conf_totals[name] += cm

        keep = test[["week_start_date"]].copy()
        keep["y"] = y_te.values
        keep["p"] = y_prob
        oof[name].append(keep)

        # A round with only one class in test can't produce an AUC.
        both_classes = y_te.nunique() == 2

        fold_rows.append({
            "round": r,
            "model": name,
            "n_train": len(train), "n_test": len(test),
            "test_positives": int(y_te.sum()),
            "cutoff": round(thr, 2),
            "TP": tp, "FN": fn, "FP": fp, "TN": tn,
            "recall": recall_score(y_te, y_pred, zero_division=0),
            "precision": precision_score(y_te, y_pred, zero_division=0),
            "f1": f1_score(y_te, y_pred, zero_division=0),
            "auc": roc_auc_score(y_te, y_prob) if both_classes else np.nan,
        })

fold_scores = pd.DataFrame(fold_rows)

print("\n" + "=" * 84)
print("PER-ROUND SCORES")
print("=" * 84)
for name in fold_scores["model"].unique():
    sub = fold_scores[fold_scores["model"] == name]
    print(f"\n{name}")
    print(sub[["round", "n_train", "n_test", "test_positives", "cutoff",
               "TP", "FN", "FP", "TN",
               "recall", "precision", "f1", "auc"]].round(3).to_string(index=False))


# ══════════════════════════════════════════════════════════════════════
# PART 4 — Combine the rounds into one scorecard per model
# ══════════════════════════════════════════════════════════════════════
# mean  = the honest estimate of performance
# std   = stability. Similar scores round to round -> trustworthy.
#         Wild swings -> treat the mean with suspicion (small test
#         blocks make this common).

METRICS = ["recall", "precision", "f1", "auc"]

summary = fold_scores.groupby("model")[METRICS].agg(["mean", "std"]).round(3)
summary.columns = [f"{m}_{s}" for m, s in summary.columns]

totals = pd.DataFrame([
    {"model": n,
     "total_TP": cm[1, 1], "total_FN": cm[1, 0],
     "total_FP": cm[0, 1], "total_TN": cm[0, 0]}
    for n, cm in conf_totals.items()
]).set_index("model")

scorecard = summary.join(totals)

print("\n" + "=" * 84)
print("AVERAGED SCORECARD  (mean +/- std across rounds)")
print("=" * 84)
print(scorecard.to_string())

print("\nCombined confusion matrices (all rounds summed):")
for name, cm in conf_totals.items():
    print(f"\n  {name}")
    print("                   predicted_no   predicted_trend")
    print(f"    actual_no      {cm[0,0]:>12}   {cm[0,1]:>15}")
    print(f"    actual_trend   {cm[1,0]:>12}   {cm[1,1]:>15}   <- missed trends: {cm[1,0]}")


# ══════════════════════════════════════════════════════════════════════
# PART 5 — Pick the model to deploy
# ══════════════════════════════════════════════════════════════════════
# Selection is on F1, NOT AUC. AUC scores how well the model ranks every
# row; the deliverable is the flagged set, which is what F1 measures.
# Selecting on AUC previously picked a model with worse precision AND
# worse recall than the runner-up.
#
# Recall is the priority, so a model that fails the floor is excluded
# outright — F1 alone would happily trade recall for precision.

eligible = scorecard[scorecard["recall_mean"] >= MIN_RECALL - 0.05]
if eligible.empty:
    eligible = scorecard                      # nothing met the floor; rank all
chosen = eligible["f1_mean"].idxmax()

print("\n" + "=" * 84)
print(f"Models holding recall >= {MIN_RECALL - 0.05:.2f}: {eligible.index.tolist()}")
print(f"CHOSEN (best F1 among them): {chosen}")
print("=" * 84)


# ══════════════════════════════════════════════════════════════════════
# PART 6 — The precision / recall trade-off
# ══════════════════════════════════════════════════════════════════════
# You cannot raise recall and precision together by moving the cutoff —
# you slide along this curve. Two ways to pick an operating point:
#
#   probability cutoff — flag everything above p. Simple, but the number
#                        of flags per week varies.
#   top-K per week     — flag the K highest-scoring categories each week.
#                        The label itself is "top 20% of this week", so a
#                        fixed K matches its shape and gives the client a
#                        predictable shortlist size.

T = pd.concat(oof[chosen])
per_week_pos = T.groupby("week_start_date")["y"].sum().mean()
print(f"\n{chosen}: {len(T)} pooled test rows, {int(T.y.sum())} real trends "
      f"(base rate {T.y.mean():.2f}, ~{per_week_pos:.1f} per week)")

print(f"\n{'cutoff':>8}{'flagged':>9}{'recall':>8}{'precision':>11}{'F1':>7}{'FN':>5}{'FP':>5}")
for t in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    yp = (T.p >= t).astype(int)
    cm = confusion_matrix(T.y, yp, labels=[0, 1])
    print(f"{t:>8.2f}{yp.sum():>9}{recall_score(T.y,yp,zero_division=0):>8.2f}"
          f"{precision_score(T.y,yp,zero_division=0):>11.2f}"
          f"{f1_score(T.y,yp,zero_division=0):>7.2f}{cm[1,0]:>5}{cm[0,1]:>5}")

print(f"\n{'top-K':>8}{'flagged':>9}{'recall':>8}{'precision':>11}{'F1':>7}{'FN':>5}{'FP':>5}")
for K in [3, 4, 5, 6, 7]:
    yp = (T.groupby("week_start_date")["p"]
           .rank(ascending=False, method="first") <= K).astype(int)
    cm = confusion_matrix(T.y, yp, labels=[0, 1])
    print(f"{K:>8}{yp.sum():>9}{recall_score(T.y,yp,zero_division=0):>8.2f}"
          f"{precision_score(T.y,yp,zero_division=0):>11.2f}"
          f"{f1_score(T.y,yp,zero_division=0):>7.2f}{cm[1,0]:>5}{cm[0,1]:>5}")

# Deployed operating point: the average cutoff the rounds settled on.
# Swap for TOP_K if the client wants a fixed-size weekly shortlist.
DEPLOY_CUTOFF = float(np.mean(thresholds[chosen]))
TOP_K = None          # e.g. 5 — overrides DEPLOY_CUTOFF in score_week()


# ══════════════════════════════════════════════════════════════════════
# PART 7 — Build the final deployed model
# ══════════════════════════════════════════════════════════════════════
# Train the chosen TYPE on ALL the data. The round-models were throwaway
# probes used only to produce the scorecard; this is the one that goes
# live, and it gets every week of history.
#
# There is deliberately NO test here. It has seen all the data, so any
# score would be it grading its own homework. The Part 4 scorecard IS
# the performance estimate — that's what the rounds were for.

final_model = build_models()[chosen]
if chosen == "XGBoost":
    n_pos = max(int(model_df[TARGET].sum()), 1)
    final_model.set_params(scale_pos_weight=(len(model_df) - n_pos) / n_pos)

final_model.fit(model_df[FEATURES], model_df[TARGET])

print(f"\nFinal model: {chosen} trained on all {len(model_df)} rows "
      f"({len(weeks)} weeks), cutoff {DEPLOY_CUTOFF:.2f}. "
      f"Not scored — see the scorecard above.")

# What the model keyed on — sanity-check against intuition
if chosen == "LogisticRegression":
    s = pd.Series(final_model.named_steps["clf"].coef_[0], index=FEATURES)
    print("\nCoefficients (positive = pushes toward 'will trend'):")
else:
    s = pd.Series(final_model.feature_importances_, index=FEATURES)
    print("\nFeature importances:")
print(s.sort_values(ascending=False).round(3).to_string())

# ── Baseline to beat ───────────────────────────────────────────────
# The label is highly autocorrelated: busy categories tend to stay busy.
# Sorting by rolling_avg alone already scores well, so quote the model's
# AUC next to this number or the gain looks bigger than it is.
print(f"\nBaseline (rank by rolling_avg alone, no model): "
      f"AUC {roc_auc_score(model_df[TARGET], model_df['rolling_avg']):.3f}")
print(f"Model mean AUC across rounds:                    "
      f"{scorecard.loc[chosen,'auc_mean']:.3f}")


# ══════════════════════════════════════════════════════════════════════
# PART 8 — Scoring a new week
# ══════════════════════════════════════════════════════════════════════
# The latest weeks have no label (their next 2 weeks haven't happened) —
# which is exactly the situation the model exists for.

def score_week(rows, top_k=TOP_K, cutoff=DEPLOY_CUTOFF):
    out = rows[["food_category", "total_posts"]].copy()
    out["trend_probability"] = final_model.predict_proba(rows[FEATURES])[:, 1].round(3)
    if top_k:
        out["flagged"] = (out["trend_probability"]
                          .rank(ascending=False, method="first") <= top_k).astype(int)
    else:
        out["flagged"] = (out["trend_probability"] >= cutoff).astype(int)
    return out.sort_values("trend_probability", ascending=False)


latest = model_df[model_df["week_start_date"] == model_df["week_start_date"].max()]
print(f"\nExample — {pd.Timestamp(latest['week_start_date'].iloc[0]).date()}:")
print(score_week(latest).head(8).to_string(index=False))

# import joblib
# joblib.dump({"model": final_model, "features": FEATURES,
#              "cutoff": DEPLOY_CUTOFF, "top_k": TOP_K}, "trend_model.pkl")

In [ ]:
# Predict for the most recent week
latest_week = weekly_volume["week_start_date"].max()
latest_df   = weekly_volume[weekly_volume["week_start_date"] == latest_week].copy()
latest_df[FEATURES] = latest_df[FEATURES].fillna(0)

if final_model is not None:
    latest_df["trend_probability"] = final_model.predict_proba(latest_df[FEATURES])[:,1]
    latest_df["prediction"]        = final_model.predict(latest_df[FEATURES])
else:
    # Fallback: composite score from growth_rate + platform_count
    latest_df["trend_probability"] = (
        latest_df["growth_rate"].clip(0, 2) / 2 * 0.6 +
        (latest_df["platform_count"] / latest_df["platform_count"].max().clip(1)) * 0.4
    ).clip(0, 1)
    latest_df["prediction"] = (latest_df["trend_probability"] >= 0.5).astype(int)

latest_df["prediction_week"] = latest_week

results = latest_df.sort_values("trend_probability", ascending=False)

# write_to_db(
#     results[["food_category","topic_label","total_posts","growth_rate",
#              "platform_count","trend_probability",
#              "prediction","prediction_week"]],
#     "trend_predictions", if_exists="replace"
# )

print(f"\n  FOOD TREND PREDICTIONS — week of {latest_week.date()}")
print(f"  {'Topic':<35} {'Posts':>6} {'Growth':>8} {'P(trend)':>10} {'Trending?':>10}")
print("  " + "-"*72)
print(results)




In [ ]:
results.drop(columns=["week_start_date","future_avg"], inplace=True)

In [ ]:
display(results)

In [ ]:
write_to_db(results, "trend_predictions", if_exists="replace")